# RAG AI Analyst
A simple retrieval-augmented generation chatbot for ecommerce analysis.


## 1. Business Problem
Create an AI analyst that answers ecommerce questions from products, reviews, policies, FAQs, and dashboard metrics.


## 2. Business Objectives
- Build a searchable knowledge base
- Retrieve relevant context using semantic search
- Generate grounded analyst answers


## 3. RAG Pipeline
Documents ? Embeddings ? Vector Database ? Retriever ? GPT Prompt ? Answer


In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

try:
    import faiss
except ImportError:
    faiss = None


## 4. Load Dataset


In [2]:
master_df = pd.read_parquet(Path("../data/processed/master_df.parquet"))
master_df.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_name,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,...,1,paypal,1,2084.010010,None,NaN,None,None,NaT,NaT
1,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,...,1,paypal,1,2084.010010,None,NaN,None,None,NaT,NaT
2,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaT,NaT,2026-01-04 08:33:20,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,...,1,paypal,1,2084.010010,None,NaN,None,None,NaT,NaT
3,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,...,1,credit_card,6,1406.069946,0ba49642-47bb-4817-9cc2-bb9ded7de7c5,5.0,Perfect,The quality is amazing. Delivery was quick.,2019-06-19 05:08:44,2019-06-19 07:08:44
4,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,...,1,credit_card,6,1406.069946,0ba49642-47bb-4817-9cc2-bb9ded7de7c5,5.0,Perfect,The quality is amazing. Delivery was quick.,2019-06-19 05:08:44,2019-06-19 07:08:44


## 5. Dataset Overview


In [3]:
print("Dataset shape:", master_df.shape)
print("Available columns:", master_df.columns.tolist())


Dataset shape: (2530433, 49)
Available columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_name', 'customer_gender', 'customer_age', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_segment', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price_x', 'freight_value', 'discount_rate', 'product_category_name', 'product_name', 'product_brand', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'cost', 'price_y', 'seller_company_name', 'seller_contact_name', 'seller_contact_gender', 'seller_contact_age', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', '

## 6. Knowledge Base Sources


In [4]:
print("Knowledge base sources:")
print("1. Products")
print("2. Customer reviews")
print("3. Policies and FAQs")
print("4. Dashboard metrics")


Knowledge base sources:
1. Products
2. Customer reviews
3. Policies and FAQs
4. Dashboard metrics


## 7. Product Documents


In [5]:
product_docs = (master_df.groupby("product_id")
    .agg(Category=("product_category_name", "first"), Seller=("seller_id", "first"), Price=("payment_value", "mean"))
    .reset_index())
product_docs["text"] = product_docs.apply(
    lambda x: f"Product {x['product_id']} belongs to category {x['Category']}, is sold by {x['Seller']}, and has average price {x['Price']:.2f}.", axis=1
)
product_docs["source"] = "Product"


## 8. Review Documents


In [6]:
review_docs = master_df.dropna(subset=["review_comment_message"])[["product_id", "seller_id", "review_score", "review_comment_message"]].copy()
review_docs = review_docs.drop_duplicates(subset=["review_comment_message"]).head(5000)
review_docs["text"] = review_docs.apply(
    lambda x: f"Review for product {x['product_id']} from seller {x['seller_id']}. Rating: {x['review_score']}. Review: {x['review_comment_message']}", axis=1
)
review_docs["source"] = "Review"


## 9. Policy and FAQ Documents


In [7]:
policy_docs = pd.DataFrame({
    "text": [
        "FAQ: Delivery dates depend on seller processing time and customer location.",
        "FAQ: Refund requests should be raised through the order support process.",
        "Policy: Product reviews reflect customer opinions and may contain delivery or quality feedback.",
        "Policy: Sellers are evaluated using order, review, and delivery information."
    ],
    "source": "Policy/FAQ"
})


## 10. Dashboard Metric Documents


In [8]:
metrics_docs = pd.DataFrame({
    "text": [
        f"Dashboard metric: total revenue is {master_df['payment_value'].sum():.2f}.",
        f"Dashboard metric: total orders are {master_df['order_id'].nunique()}.",
        f"Dashboard metric: average review score is {master_df['review_score'].mean():.2f}.",
        f"Dashboard metric: total customers are {master_df['customer_unique_id'].nunique()}."
    ],
    "source": "Dashboard"
})


## 11. Combine Documents


In [9]:
documents = pd.concat([
    product_docs[["text", "source"]],
    review_docs[["text", "source"]],
    policy_docs,
    metrics_docs
], ignore_index=True)
documents = documents.dropna(subset=["text"]).reset_index(drop=True)
print("Knowledge-base documents:", len(documents))
display(documents.head())


Knowledge-base documents: 2657


,text,source
0,Product 00078596-b7ae-4c45-b1e8-c9a480957b28 b...,Product
1,Product 001006dd-cbbd-41b9-9b6b-3607f0ab1981 b...,Product
2,Product 001304db-9813-4ffc-bf93-8b454eae1588 b...,Product
3,Product 001e7507-54d2-438b-9da0-59f31c3dc9cc b...,Product
4,Product 003e6e7c-6357-40af-a3bb-6493d03b14c6 b...,Product


## 12. Text Preparation


In [10]:
documents["text"] = documents["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


## 13. Embedding Models


In [11]:
print("Embedding options: OpenAI, BGE, E5, and Sentence Transformers.")
print("This notebook uses Sentence Transformers when installed, otherwise TF-IDF as a local fallback.")


Embedding options: OpenAI, BGE, E5, and Sentence Transformers.
This notebook uses Sentence Transformers when installed, otherwise TF-IDF as a local fallback.


## 14. Create Embeddings


In [12]:
if SentenceTransformer is not None:
    embedding_model_name = "all-MiniLM-L6-v2"
    embedding_model = SentenceTransformer(embedding_model_name)
    embeddings = embedding_model.encode(documents["text"].tolist(), normalize_embeddings=True)
    embedding_type = "Sentence Transformers"
else:
    embedding_model = TfidfVectorizer(max_features=10000, stop_words="english")
    embeddings = embedding_model.fit_transform(documents["text"])
    embedding_type = "TF-IDF fallback"

print("Embedding model:", embedding_type)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: Sentence Transformers


## 15. Vector Database


In [13]:
print("Vector DB options: FAISS, Chroma, and Pinecone.")
print("This notebook uses FAISS when available; otherwise it searches the local embedding matrix.")


Vector DB options: FAISS, Chroma, and Pinecone.
This notebook uses FAISS when available; otherwise it searches the local embedding matrix.


## 16. Build FAISS Index


In [14]:
vector_index = None
if faiss is not None and not hasattr(embeddings, "toarray"):
    vector_index = faiss.IndexFlatIP(embeddings.shape[1])
    vector_index.add(np.asarray(embeddings, dtype="float32"))
    print("FAISS index built.")
else:
    print("Using local cosine-similarity retrieval.")


FAISS index built.


## 17. Semantic Search


In [15]:
def retrieve(query, k=5):
    if SentenceTransformer is not None:
        query_embedding = embedding_model.encode([query], normalize_embeddings=True)
        if vector_index is not None:
            scores, indices = vector_index.search(np.asarray(query_embedding, dtype="float32"), k)
            results = documents.iloc[indices[0]].copy()
            results["score"] = scores[0]
            return results
        scores = cosine_similarity(query_embedding, embeddings).ravel()
    else:
        query_embedding = embedding_model.transform([query])
        scores = cosine_similarity(query_embedding, embeddings).ravel()
    results = documents.copy()
    results["score"] = scores
    return results.nlargest(k, "score")


## 18. Retriever Test


In [16]:
query = "What are the main customer complaints about delivery?"
retrieved_docs = retrieve(query, k=5)
display(retrieved_docs)


,text,source,score
2636,Review for product 7f0ec0cb-2611-4057-9569-7f5...,Review,0.464202
2652,"Policy: Sellers are evaluated using order, rev...",Policy/FAQ,0.462368
2651,Policy: Product reviews reflect customer opini...,Policy/FAQ,0.445174
2645,Review for product f3c71716-ebdf-4e99-bced-409...,Review,0.441134
2203,Review for product 29b1672f-2118-49f8-b158-955...,Review,0.439576


## 19. RAG Prompt Template


In [17]:
def build_rag_prompt(question, context_docs):
    context = "\n".join(f"[{row.source}] {row.text}" for row in context_docs.itertuples())
    return f"""You are an ecommerce AI analyst.
Answer the question using only the supplied context.
If the context does not contain the answer, say that clearly.
Keep the answer concise and cite the source type in brackets.

Context:
{context}

Question: {question}
Answer:"""


## 20. GPT Answer Generation


In [18]:
question = "What should the business improve based on customer feedback?"
context_docs = retrieve(question, k=5)
prompt = build_rag_prompt(question, context_docs)
print(prompt)
print("Send this prompt to GPT through your approved OpenAI API integration to generate the final answer.")


You are an ecommerce AI analyst.
Answer the question using only the supplied context.
If the context does not contain the answer, say that clearly.
Keep the answer concise and cite the source type in brackets.

Context:
[Policy/FAQ] Policy: Product reviews reflect customer opinions and may contain delivery or quality feedback.
[Dashboard] Dashboard metric: total customers are 279199.
[Review] Review for product b0e19dea-6966-461e-884b-a93b34a76f0c from seller b6918492-3095-4302-9216-902a798facab. Rating: 4.0. Review: The service is perfect. Delivery was quick. Good for the price.
[Review] Review for product b3f39816-6be7-4f4c-b1c2-bf427cf3a304 from seller b381e83b-e28e-4eee-b295-4178fee20567. Rating: 4.0. Review: The service is amazing. Delivery was quick.
[Review] Review for product cc646824-ec6b-41ba-8d71-8f626cb45dfc from seller 336344b5-f9b0-4d34-b4c4-3eec1aead941. Rating: 4.0. Review: The service is good. Arrived early! Great value.

Question: What should the business improve base

## 21. LLM Options


In [19]:
print("GPT can generate final answers from retrieved context.")
print("Llama and other local models can use the same prompt template.")
print("Keep the retrieved context with every LLM call to reduce hallucinations.")


GPT can generate final answers from retrieved context.
Llama and other local models can use the same prompt template.
Keep the retrieved context with every LLM call to reduce hallucinations.


## 22. LangChain and LlamaIndex


In [20]:
print("LangChain and LlamaIndex can orchestrate document loading, retrieval, and LLM calls.")
print("This notebook keeps the RAG flow explicit and dependency-light for learning.")


LangChain and LlamaIndex can orchestrate document loading, retrieval, and LLM calls.
This notebook keeps the RAG flow explicit and dependency-light for learning.


## 23. Product Search Example


In [21]:
display(retrieve("Which products and categories have strong customer ratings?", k=5))


,text,source,score
2651,Policy: Product reviews reflect customer opini...,Policy/FAQ,0.537606
2053,Review for product d1c12310-e34f-4074-bd22-ee4...,Review,0.431565
2020,Review for product 775fda99-84ae-4a0d-b9b8-917...,Review,0.431060
2656,Dashboard metric: total customers are 279199.,Dashboard,0.422785
2609,Review for product e9549f32-db53-4023-b624-db8...,Review,0.422183


## 24. Review Search Example


In [22]:
display(retrieve("Show review feedback about product quality and delivery delays.", k=5))


,text,source,score
2651,Policy: Product reviews reflect customer opini...,Policy/FAQ,0.706570
2645,Review for product f3c71716-ebdf-4e99-bced-409...,Review,0.513400
2040,Review for product 415911fe-34d5-405c-abdb-dd6...,Review,0.498587
2030,Review for product 0be8ed49-fc0e-4841-b19f-a66...,Review,0.498376
2138,Review for product e37a48a3-3ea8-4563-891c-d3e...,Review,0.483599


## 25. Policy Search Example


In [23]:
display(retrieve("How should customers request a refund?", k=5))


,text,source,score
2650,FAQ: Refund requests should be raised through ...,Policy/FAQ,0.666548
2651,Policy: Product reviews reflect customer opini...,Policy/FAQ,0.337351
2652,"Policy: Sellers are evaluated using order, rev...",Policy/FAQ,0.327638
2656,Dashboard metric: total customers are 279199.,Dashboard,0.251497
2111,Review for product 38e36858-4ba7-4411-b1b5-fcb...,Review,0.245037


## 26. Dashboard Search Example


In [24]:
display(retrieve("What are the total revenue and customer metrics?", k=5))


,text,source,score
2653,Dashboard metric: total revenue is 3294778880.00.,Dashboard,0.685187
2656,Dashboard metric: total customers are 279199.,Dashboard,0.655980
2654,Dashboard metric: total orders are 1000000.,Dashboard,0.502604
2655,Dashboard metric: average review score is 3.98.,Dashboard,0.325902
403,Product 369257dc-0a43-498f-919c-0ef30ffd1df6 b...,Product,0.291677


## 27. Grounded Answer Checks


In [25]:
def answer_with_sources(question, k=5):
    context = retrieve(question, k)
    return {"question": question, "context": context[["source", "text", "score"]], "prompt": build_rag_prompt(question, context)}

answer_package = answer_with_sources("Summarise seller performance risks.")
display(answer_package["context"])


,source,text,score
2652,Policy/FAQ,"Policy: Sellers are evaluated using order, rev...",0.557003
2651,Policy/FAQ,Policy: Product reviews reflect customer opini...,0.358459
2645,Review,Review for product f3c71716-ebdf-4e99-bced-409...,0.325685
2593,Review,Review for product ea852ab1-ddf0-4401-91b3-7e6...,0.325196
2246,Review,Review for product adf3b13b-5a78-445c-9b63-b49...,0.320611


## 28. RAG Evaluation


In [26]:
print("Simple RAG checks:")
print("- Retrieval relevance: inspect top returned documents.")
print("- Groundedness: answer only from the returned context.")
print("- Completeness: confirm key source types are represented.")


Simple RAG checks:
- Retrieval relevance: inspect top returned documents.
- Groundedness: answer only from the returned context.
- Completeness: confirm key source types are represented.


## 29. Save Retriever Artifacts


In [27]:
# Create models directory
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save retriever artifacts
model_path = MODEL_DIR / "rag_retriever.joblib"
retriever_artifacts = {"documents": documents, "embeddings": embeddings, "embedding_type": embedding_type}
if embedding_type == "TF-IDF fallback":
    retriever_artifacts["vectorizer"] = embedding_model
else:
    retriever_artifacts["embedding_model_name"] = embedding_model_name
joblib.dump(retriever_artifacts, model_path)

print("=" * 60)
print(f"Embedding Model : {embedding_type}")
print("Retriever saved successfully!")
print(f"Location : {model_path}")
print("=" * 60)


Embedding Model : Sentence Transformers
Retriever saved successfully!
Location : ..\models\rag_retriever.joblib


## 30. Deployment Recommendation


In [28]:
print("Use FAISS for local vector search, Chroma for a local persistent store, or Pinecone for managed scale.")
print("Connect the retriever to GPT only after retrieving relevant context.")


Use FAISS for local vector search, Chroma for a local persistent store, or Pinecone for managed scale.
Connect the retriever to GPT only after retrieving relevant context.


## 31. Executive Summary


In [29]:
print("RAG AI Analyst workflow completed.")
print(f"Knowledge-base documents: {len(documents)}")
print(f"Embedding approach: {embedding_type}")
print("The system can retrieve relevant ecommerce context before generating a grounded LLM answer.")


RAG AI Analyst workflow completed.
Knowledge-base documents: 2657
Embedding approach: Sentence Transformers
The system can retrieve relevant ecommerce context before generating a grounded LLM answer.
